In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% ! important;}
div.cell.code_cell.rendered{width:100%}
div.input_prompt{padding:0px}
div.CodeMirror {font-family:Consolas ; font-size:12pt;}
div.text_cell_render.rendered_html {font-size:12pt;}
div.output {font-size:12pt; font-weight:bold}
div.input {font-family:Consolas ; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper {padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))


<b><font size="6" color="red">ch.15 데이터 베이스 연동</font></b>
# 1절 SQLite 데이터 베이스 연결
- SQLite 데이터 베이스는 별도의 DBMS 없이 SQL문 이용해서 DB 엑세스 할 수 있도록 만든 간단한 디스크 기반의 DB
- C라이브러리
- SQLite는 프로토타입 생성 시 사용
- 프로젝트 : 분석 ->    설계    -> 구현 ->    테스트     ->    고객에게 배포
                 프로토타입(시제품)   완제품(Oracle, MySQL, Maria,...)
- [DB browser fo SQLite](https://sqlitebrowser.org/)
## 1.1 SQLite brower 설치 및 sqlite 패키지 load

In [8]:
import sqlite3
sqlite3.sqlite_version # 버전

'3.40.1'

In [9]:
import pandas as pd
pd.__version__

'1.5.3'

## 1.2 데이터 베이스 연결
```
(1) 데이터베이스 연결
(2) SQL 전송 객체 생성(cursor)
(3) SQL 전송 & 결과 받기(cursor.excute()이용)
(4) cursor 해제 & 데이터 베이스 연결객체 해제
```

In [4]:
# (1) DB 연결 : sqlite로 연결시 DB파일이 없으면 빈 DB파일 생성, 있으면 연결
conn = sqlite3.connect('data/ch15_example.db')
conn

In [5]:
# (2) SQL 전송 객체 생성(cursor 객체)
cursor = conn.cursor()
cursor

In [7]:
cursor.execute('''
    CREATE TABLE MEMBER(
        NAME TEXT,
        AGE INT,
        EMAIL TEXT
    )
''')

In [8]:
cursor.execute('DROP TABLE MEMBER')

In [9]:
cursor.execute('''
    CREATE TABLE MEMBER(
        NAME TEXT,
        AGE INT,
        EMAIL TEXT
    )
''')

In [11]:
cursor.execute('INSERT INTO MEMBER VALUES("홍길동",20,"h@h.com")')
print('수행결과 행 수 :', cursor.rowcount)

수행결과 행 수 : 1


In [13]:
sql = 'INSERT INTO MEMBER VALUES("김길동",30,"K@K.COM")'
cursor.execute(sql)
print('수행결과 행 수 :', cursor.rowcount)
cursor.execute('INSERT INTO MEMBER VALUES("이길동",25,"l@l.com")')
print('수행결과 행 수 :', cursor.rowcount)

수행결과 행 수 : 1
수행결과 행 수 : 1


In [14]:
conn.commit() # (<>)conn.rollback()

In [17]:
# select 전송 결과 : cursor가 가리킴
cursor.execute('SELECT * FROM MEMBER')

In [18]:
# insert, update, delete문 실행 결과: cursor.rowcount
# select문 실행 결과 : 
        ## fetchall() : 결과를 모두 받을 때(튜플 list)
        ## fetchone() : 결과를 한 행씩 받을 때(튜플)
        ## fetchmany(n) : 결과를 n행 받을 때(튜플 list)
print(cursor.fetchall())

[('홍길동', 20, 'h@h.com'), ('김길동', 30, 'K@K.COM'), ('이길동', 25, 'l@l.com')]


In [19]:
print(cursor.fetchall()) # 한 번 소요된 curswor객체는 다시 fetch할 수 없음

[]


In [21]:
# 모든 줄 한번에 읽어오기
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE")
members = cursor.fetchall()
members

[('홍길동', 20, 'h@h.com'), ('이길동', 25, 'l@l.com'), ('김길동', 30, 'K@K.COM')]

In [22]:
for member in members:
    print(member)

('홍길동', 20, 'h@h.com')
('이길동', 25, 'l@l.com')
('김길동', 30, 'K@K.COM')


In [24]:
# 한 줄씩 읽기
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE")
members= []
while True:
    member = cursor.fetchone()
    if member is None:
        print('데이터 끝')
        break
    members.append(member)

데이터 끝


In [25]:
members

[('홍길동', 20, 'h@h.com'), ('이길동', 25, 'l@l.com'), ('김길동', 30, 'K@K.COM')]

In [26]:
# 최상위 n행 읽기
cursor.execute("SELECT * FROM MEMBER ORDER BY AGE")
for member in cursor.fetchmany(2):
    print(member)

('홍길동', 20, 'h@h.com')
('이길동', 25, 'l@l.com')


In [27]:
cursor.description

(('NAME', None, None, None, None, None, None),
 ('AGE', None, None, None, None, None, None),
 ('EMAIL', None, None, None, None, None, None))

In [33]:
class Member:
    'member 테이블의 내용을 받을 객체 타입'
    def __init__(self, name,age, email):
        self.name = name
        self.age = age
        self.email = email
    def __str__(self):
        return f"{self.name}\t{self.age}\t{self.email}"
def to_member(*row): # 튜플매개변수를 받아 member형 객체를 return
    return Member(row[0], row[1], row[2])

In [34]:
member = to_member('홍길동', 20, 'h@h.con')
print(member)

홍길동	20	h@h.con


In [37]:
cursor.execute('SELECT * FROM MEMBER')
member_list = [] # sql문 수행한 결과를 담을 객체 list
members = cursor.fetchall() # 튜플 list
# print(members)
for member in members:
    member_list.append(to_member(*member))

In [38]:
for member in member_list:
    print(member)

홍길동	20	h@h.com
김길동	30	K@K.COM
이길동	25	l@l.com


In [39]:
# (4) 연결 해제
cursor.close()
conn.close()

## 1.3 SQL 구문에 파라미터 사용하기
- qmark (DB에 따라 불가한 경우 있음) 
- named (추천)

In [41]:
conn = sqlite3.connect('data/ch15_example.db')
cursor = conn.cursor()
cursor.execute('SELECT * FROM MEMBER WHERE NAME IN("홍길동","김길동")')
cursor.fetchall()

[('홍길동', 20, 'h@h.com'), ('김길동', 30, 'K@K.COM')]

In [46]:
# 파라미터 사용하기 : qmark 방법
name1 = input('검색할 이름1?')
name2 = input('검색할 이름2?')
# cursor.execute('SELECT * FROM MEMBER WHERE NAME IN("'+name1+'","'+name2+'")')
# cursor.execute(f'SELECT * FROM MEMBER WHERE NAME IN("{name1}","{name2}")')
cursor.execute('SELECT * FROM MEMBER WHERE NAME IN(?,?)', (name1, name2))
cursor.fetchall()

검색할 이름1?홍길동
검색할 이름2?김길동


[('홍길동', 20, 'h@h.com'), ('김길동', 30, 'K@K.COM')]

In [48]:
# 파라미터 사용하기 : named 방법
name1 = input('검색할 이름1?')
name2 = input('검색할 이름2?')
cursor.execute('SELECT * FROM MEMBER WHERE NAME IN(:name1,:name2)', {'name1' : name1, 'name2' : name2})
cursor.fetchall()

검색할 이름1?홍길동
검색할 이름2?이딜동


[('홍길동', 20, 'h@h.com')]

In [51]:
# member 테이블에 입력(사용자로부터 이름, 나이, 메일을 받아 insert)
try:
    name = input('입력할 이름 :')
    age = int(input('입력할 나이 :'))
except:
    print('유효하지 않은 나이를 입력한 경우, 18세로 초기화')
    age = 18
finally:
    email = input('입력할 메일은 :')
inputdata = {'name':name,'age':age,'email':email}  # named 방식
inputdata2 = (name, age, email) #qmark 방식
sql = "INSERT INTO MEMBER VALUES(:name,:age,:email)"
cursor.execute(sql,inputdata)
conn.commit()
if cursor.rowcount:
    print('저장완료')

입력할 이름 :마길동
입력할 나이 :ㅁ
유효하지 않은 나이를 입력한 경우, 18세로 초기화
입력할 메일은 :
저장완료


In [61]:
while True:
    try:
        name = input('입력할 이름(종료는 0) :')
        if name == '0':
            break;
        age = int(input('입력할 나이 :'))
    except:
        print('유효하지 않은 나이를 입력한 경우, 18세로 초기화')
        age = 18
    email = input('입력할 메일은 :')
    newMember = Member(name,age,email)
    # print(newMember)
    # print(newMember.__dict__)
    sql = 'INSERT INTO MEMBER VALUES(:name, :age, :email)'
    cursor.execute(sql, newMember.__dict__)
    if cursor.rowcount==1:
        print('입력 성공')
conn.commit()

입력할 이름(종료는 0) :0


In [62]:
cursor.close()
conn.close()

# 2절 오라클 데이터 베이스 연결
- pip install cx_oracle (oracle 11g까지)
- pip install oracledb (oracle 12부터)

In [10]:
import cx_Oracle
cx_Oracle.__version__

ModuleNotFoundError: No module named 'cx_Oracle'

In [67]:
# conn 얻어오는 방법1
conn = cx_Oracle.connect("scott","tiger", 'localhost:1521/xe')
cursor = conn.cursor()
sql = 'SELECT EMPNO "NO", ENAME, JOB, HIREDATE, SAL, COMM, DEPTNO FROM EMP'
cursor.execute(sql)
emp = cursor.fetchall()
print(emp)

[(7369, 'SMITH', 'CLERK', datetime.datetime(1980, 12, 17, 0, 0), 800.0, None, 20), (7499, 'ALLEN', 'SALESMAN', datetime.datetime(1981, 2, 20, 0, 0), 1600.0, 300.0, 30), (7521, 'WARD', 'SALESMAN', datetime.datetime(1981, 2, 22, 0, 0), 1250.0, 500.0, 30), (7566, 'JONES', 'MANAGER', datetime.datetime(1981, 4, 2, 0, 0), 2975.0, None, 20), (7654, 'MARTIN', 'SALESMAN', datetime.datetime(1981, 9, 28, 0, 0), 1250.0, 1400.0, 30), (7698, 'BLAKE', 'MANAGER', datetime.datetime(1981, 5, 1, 0, 0), 2850.0, None, 30), (7782, 'CLARK', 'MANAGER', datetime.datetime(1981, 6, 9, 0, 0), 2450.0, None, 10), (7788, 'SCOTT', 'ANALYST', datetime.datetime(1982, 12, 9, 0, 0), 3000.0, None, 20), (7839, 'KING', 'PRESIDENT', datetime.datetime(1981, 11, 17, 0, 0), 5000.0, None, 10), (7844, 'TURNER', 'SALESMAN', datetime.datetime(1981, 9, 8, 0, 0), 1500.0, 0.0, 30), (7876, 'ADAMS', 'CLERK', datetime.datetime(1983, 1, 12, 0, 0), 1100.0, None, 20), (7900, 'JAMES', 'CLERK', datetime.datetime(1981, 12, 3, 0, 0), 950.0, Non

In [11]:
# conn 얻어오는 방법2
import oracledb
oracledb.init_oracle_client()
# conn = oracledb.connect("scott/tiger@localhost:1521/xe")
conn = oracledb.connect(
    user="scott",
    password='tiger',
    host='localhost',
    port=1521,
    sid='xe'
)
cursor = conn.cursor()
sql = 'SELECT EMPNO "NO", ENAME, JOB, HIREDATE, SAL, COMM, DEPTNO FROM EMP'
cursor.execute(sql)
emp = cursor.fetchall()
print(emp)

[(7369, 'SMITH', 'CLERK', datetime.datetime(1980, 12, 17, 0, 0), 800.0, None, 20), (7499, 'ALLEN', 'SALESMAN', datetime.datetime(1981, 2, 20, 0, 0), 1600.0, 300.0, 30), (7521, 'WARD', 'SALESMAN', datetime.datetime(1981, 2, 22, 0, 0), 1250.0, 500.0, 30), (7566, 'JONES', 'MANAGER', datetime.datetime(1981, 4, 2, 0, 0), 2975.0, None, 20), (7654, 'MARTIN', 'SALESMAN', datetime.datetime(1981, 9, 28, 0, 0), 1250.0, 1400.0, 30), (7698, 'BLAKE', 'MANAGER', datetime.datetime(1981, 5, 1, 0, 0), 2850.0, None, 30), (7782, 'CLARK', 'MANAGER', datetime.datetime(1981, 6, 9, 0, 0), 2450.0, None, 10), (7788, 'SCOTT', 'ANALYST', datetime.datetime(1982, 12, 9, 0, 0), 3000.0, None, 20), (7839, 'KING', 'PRESIDENT', datetime.datetime(1981, 11, 17, 0, 0), 5000.0, None, 10), (7844, 'TURNER', 'SALESMAN', datetime.datetime(1981, 9, 8, 0, 0), 1500.0, 0.0, 30), (7876, 'ADAMS', 'CLERK', datetime.datetime(1983, 1, 12, 0, 0), 1100.0, None, 20), (7900, 'JAMES', 'CLERK', datetime.datetime(1981, 12, 3, 0, 0), 950.0, Non

In [3]:
import pandas as pd
emp_df =pd.DataFrame(emp)
emp_df

,0,1,2,3,4,5,6
0,7369,SMITH,CLERK,1980-12-17,800.0,NaN,20
1,7499,ALLEN,SALESMAN,1981-02-20,1600.0,300.0,30
2,7521,WARD,SALESMAN,1981-02-22,1250.0,500.0,30
3,7566,JONES,MANAGER,1981-04-02,2975.0,NaN,20
4,7654,MARTIN,SALESMAN,1981-09-28,1250.0,1400.0,30
5,7698,BLAKE,MANAGER,1981-05-01,2850.0,NaN,30
6,7782,CLARK,MANAGER,1981-06-09,2450.0,NaN,10
7,7788,SCOTT,ANALYST,1982-12-09,3000.0,NaN,20
8,7839,KING,PRESIDENT,1981-11-17,5000.0,NaN,10
9,7844,TURNER,SALESMAN,1981-09-08,1500.0,0.0,30


In [10]:
# select문을 수행한 필드 정보 (no, Ename, job,~)
cursor.description

[('NO', <DbType DB_TYPE_NUMBER>, 5, None, 4, 0, False),
 ('ENAME', <DbType DB_TYPE_VARCHAR>, 10, 10, None, None, True),
 ('JOB', <DbType DB_TYPE_VARCHAR>, 9, 9, None, None, True),
 ('HIREDATE', <DbType DB_TYPE_DATE>, 23, None, None, None, True),
 ('SAL', <DbType DB_TYPE_NUMBER>, 11, None, 7, 2, True),
 ('COMM', <DbType DB_TYPE_NUMBER>, 11, None, 7, 2, True),
 ('DEPTNO', <DbType DB_TYPE_NUMBER>, 3, None, 2, 0, True)]

In [11]:
[description[0] for description in cursor.description]

['NO', 'ENAME', 'JOB', 'HIREDATE', 'SAL', 'COMM', 'DEPTNO']

In [13]:
emp_df.columns=[description[0] for description in cursor.description]
emp_df.head()

,NO,ENAME,JOB,HIREDATE,SAL,COMM,DEPTNO
0,7369,SMITH,CLERK,1980-12-17,800.0,NaN,20
1,7499,ALLEN,SALESMAN,1981-02-20,1600.0,300.0,30
2,7521,WARD,SALESMAN,1981-02-22,1250.0,500.0,30
3,7566,JONES,MANAGER,1981-04-02,2975.0,NaN,20
4,7654,MARTIN,SALESMAN,1981-09-28,1250.0,1400.0,30


In [29]:
# 검색할 이름을 사용자에게 받아 해당 내용을 출력
ename = input('검색할 이름은?').upper()
# print(ename)
sql = 'SELECT * FROM EMP WHERE ENAME = :ename'
cursor.execute(sql,{'ename':ename})
emp=cursor.fetchall()
if emp:
    print(emp[0])
else:
    print('입력하신 데이터는 없습니다.')

검색할 이름은?scott
(7788, 'SCOTT', 'ANALYST', 7566, datetime.datetime(1982, 12, 9, 0, 0), 3000.0, None, 20)


In [30]:
fieldnames = [descript[0] for descript in cursor.description]
fieldnames

['EMPNO', 'ENAME', 'JOB', 'MGR', 'HIREDATE', 'SAL', 'COMM', 'DEPTNO']

In [31]:
for fieldnames, data in zip(fieldnames, emp[0]):
    print("{}:{}".format(fieldnames, data if data is not None else ' - '))

EMPNO:7788
ENAME:SCOTT
JOB:ANALYST
MGR:7566
HIREDATE:1982-12-09 00:00:00
SAL:3000.0
COMM: - 
DEPTNO:20


In [12]:
cursor.close()
conn.close()

# 3절 MySQL 데이터베이스 연결
- pip install mysql-connector-python (공식 connector)
- pip install PyMySQL (경량)

In [33]:
%pip install PyMySQL

     ---------------------------------------- 45.3/45.3 kB ? eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [42]:
import pymysql
conn = pymysql.connect(
    host='localhost',
    user='root',
    password='mysql',
    db='devdb',
    charset='utf8mb4'
)
cursor = conn.cursor()
sql='SELECT * FROM PERSONAL'
cursor.execute(sql)
personal = cursor.fetchall()
personal
fieldnames =[descript[0] for descript in cursor.description]
fieldnames
pd.DataFrame(personal, columns=fieldnames)

,pno,pname,job,manager,startdate,pay,bonus,dno
0,1001,bill,president,NaN,1989-01-10,7000,NaN,10
1,1111,smith,manager,1001.0,1990-12-17,1000,NaN,10
2,1112,ally,salesman,1116.0,1991-02-20,1600,500.0,30
3,1113,word,salesman,1116.0,1992-02-24,1450,300.0,30
4,1114,james,manager,1001.0,1990-04-12,3975,NaN,20
5,1116,johnson,manager,1001.0,1991-05-01,3550,NaN,30
6,1118,martin,analyst,1111.0,1991-09-09,3450,NaN,10
7,1121,kim,clerk,1114.0,1990-12-08,4000,NaN,20
8,1123,lee,salesman,1116.0,1991-09-23,1200,0.0,30
9,1226,park,analyst,1111.0,1990-01-03,2500,NaN,10


In [43]:
cursor.close()
conn.close()

# 4절 연습문제
## 실습형
- 회원가입 | 전체조회 | 이름찾기 | 메일삭제 | csv보내기 | 종료

### 0. 처음실행

In [13]:
def load_cursor():
    global conn
    import oracledb
    oracledb.init_oracle_client()
    conn=oracledb.connect('scott/tiger@localhost:1521/xe')
load_cursor()

### 1. 입력

In [14]:
def fn1_insert_member_info():
    '입력받은 데이터를 member테이블에 입력'
    cursor=conn.cursor()
    name = input('이름 :')
    tel= input('전화번호 :')
    email=input('이메일 :')
    try:
        age=int(input('나이 :'))
        if age > 200:
            age = 200
        elif age<0:
            age=0
    except:
        print('유효하지 않은 나이 입력 시 나이는 0으로 초기화')
        age=0
    try:
        grade=int(input('등급 :'))
        if grade<1:
            grade=1
        elif grade >5:
            grade=5
    except:
        print('유효하지 않은 등급 입력 시 등급은 1로 초기화')
        grade=1
    etc = input('기타 정보:')
    sql = "INSERT INTO MEMBER VALUES (:name, :tel,:email, :age, :grade, :etc)"
    inputdata = {'name':name, "tel":tel, 'email':email, 'age':age, 'grade':grade, 'etc':etc}
    cursor.execute(sql, inputdata)
    conn.commit()
    if cursor.rowcount==1:
        print(name,'님 가입 완료')
    cursor.close()
# fn1_insert_member_info()

### 2,3. 조회

In [15]:
def fn2_3_print_member(chk):
    cursor=conn.cursor()
    if chk == 'all':
        sql = "SELECT * FROM MEMBER"
        cursor.execute(sql)
        to_df=cursor.fetchall()
        
    elif chk =="name":
        name=input('검색할 이름을 입력하세요 :')
        sql = 'SELECT * FROM MEMBER WHERE NAME = :name'
        cursor.execute(sql,{'name':name})
        member=cursor.fetchall()
        to_df=member
        
    if to_df:
        fieldname = [descript[0] for descript in cursor.description]
        return pd.DataFrame(to_df, columns=fieldname)
    else:
        print('입력한 회원이 없습니다.')
    cursor.close()

# fn2_3_print_member('name')

### 4. 메일로 삭제

In [16]:
def fn4_drop_members():
    cursor=conn.cursor()
    dmail = input('제거할 이메일을 입력하세요 :')
    sql = 'SELECT * FROM MEMBER WHERE NAME = :name'
    cursor.execute(sql,{'name':name})
    member=cursor.fetchall()
    finder=member

    if finder:
        sql = 'DELETE FROM MEMBER WHERE EMAIL = :damil'
        cursor.execute(sql,{'dmail': dmail})
        print('해당 이메일을 가진 데이터를 제거했습니다.')
    else:
        print('입력한 이메일이 존재하지 않습니다.')

### 5. csv파일로 저장 

In [17]:
def fn5_save_csv_member():
    cursor=conn.cursor()
    sql = "SELECT * FROM MEMBER"
    cursor.execute(sql)
    to_df=cursor.fetchall()
    fieldname = [descript[0] for descript in cursor.description]
    save_csv = pd.DataFrame(to_df, columns=fieldname)

    save_csv.to_csv(f'data/ch15_members.csv', index=False)

In [18]:
load_cursor()
while True:
    menu = input("1.회원입력 | 2.전체조회 | 3.이름찾기 | 4.메일삭제 | 5.CSV내보내기 | 0.종료")
    if menu=='1':
        fn1_insert_member_info()
    elif menu=='2':
        fn2_3_print_member('all')
    elif menu=='3':
        fn2_3_print_member('name')
    elif menu=='4':
        fn4_drop_members()
    elif menu=='5':
        fn5_save_csv_member()
    elif menu=='0':
        conn.close()
        break
    else:
        print('유효한 메뉴 번호를 입력해 주세요')

1.회원입력 | 2.전체조회 | 3.이름찾기 | 4.메일삭제 | 5.CSV내보내기 | 0.종료2
1.회원입력 | 2.전체조회 | 3.이름찾기 | 4.메일삭제 | 5.CSV내보내기 | 0.종료0
